# 07 · Spatial Analysis

Moran's I test on regression residuals + spatial lag model if significant.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'
EXT = ROOT / 'data' / 'external'
FIGS = ROOT / 'output' / 'figures'

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from spatial_utils import build_queen_weights, build_idw_weights, morans_i, plot_morans_scatter
from regression_utils import fit_model, OUTCOMES

## 7.1 Load Panel & Get Residuals from Baseline Model

In [ ]:
panel = pd.read_parquet(PROC / 'panel_for_regression.parquet')

outcome = OUTCOMES[0]  # ramp_magnitude_mwh
result_dict = fit_model(panel, outcome=outcome, model_label='model1_spatial')
result = result_dict['result']

residuals_df = pd.DataFrame({
    'zip_code': panel.set_index(['zip_code', pd.to_datetime(panel[['year', 'month']].assign(day=1))]).index.get_level_values(0),
    'residual': result.resids.values
})

# Average residual per ZIP for spatial analysis
zip_resid = residuals_df.groupby('zip_code')['residual'].mean()
print(f'Residuals across {len(zip_resid):,} ZIPs')

## 7.2 Build Spatial Weights

In [ ]:
zcta_path = EXT / 'ca_zcta.geojson'
assert zcta_path.exists(), f'ZCTA shapefile needed at {zcta_path}'

zcta_gdf = gpd.read_file(zcta_path)
zcta_gdf['zip_code'] = zcta_gdf['ZCTA5CE20'].astype(str).str.zfill(5)

# Keep only ZIPs in our panel
zcta_sub = zcta_gdf[zcta_gdf['zip_code'].isin(zip_resid.index)].copy()
zcta_sub = zcta_sub.to_crs('EPSG:3310')  # CA Albers

w_queen = build_queen_weights(zcta_sub, id_col='zip_code')
print(f'Queen weights: {w_queen.n} ZIPs, {w_queen.pct_nonzero:.2%} non-zero')

## 7.3 Moran's I Test

In [ ]:
# Align residuals to weight matrix order
aligned_resid = zip_resid.reindex(w_queen.id_order)
aligned_resid = aligned_resid.fillna(0)

mi_result = morans_i(aligned_resid, w_queen, permutations=999)
print(f"Moran's I = {mi_result['I']:.4f}")
print(f"Expected I = {mi_result['EI']:.4f}")
print(f"Z-score = {mi_result['z_score']:.3f}")
print(f"p-value (normal) = {mi_result['p_value']:.4f}")
print(f"p-value (permutation) = {mi_result['p_sim']:.4f}")

significant = mi_result['p_sim'] < 0.05
print(f"\nSpatial autocorrelation significant at p<0.05: {significant}")

In [ ]:
plot_morans_scatter(aligned_resid, w_queen)

## 7.4 Spatial Lag Model (if Moran's I significant)

In [ ]:
if significant:
    import spreg
    
    # Cross-sectional spatial lag on ZIP-averaged data
    zip_avg = panel.groupby('zip_code')[['log_btm_lag1', 'log_btm_lag1_sq', outcome]].mean().reindex(w_queen.id_order)
    y = zip_avg[outcome].fillna(0).values.reshape(-1, 1)
    X = zip_avg[['log_btm_lag1', 'log_btm_lag1_sq']].fillna(0).values
    
    sl_model = spreg.GM_Lag(y, X, w=w_queen, name_y=outcome, name_x=['log_btm_lag1', 'log_btm_lag1_sq'])
    print(sl_model.summary)
else:
    print('Spatial lag model not fitted — Moran\'s I not significant at p<0.05')

## 7.5 Coefficient Choropleth

In [ ]:
zcta_plot = zcta_sub.merge(zip_resid.reset_index().rename(columns={'residual': 'resid'}), on='zip_code', how='left')

fig, ax = plt.subplots(figsize=(10, 10))
zcta_plot.plot(column='resid', cmap='RdBu_r', legend=True, ax=ax,
               linewidth=0.1, edgecolor='white', missing_kwds={'color': 'lightgray'})
ax.set_title(f'Model Residuals by ZIP — {outcome.replace("_", " ").title()}', fontsize=13)
ax.axis('off')
plt.tight_layout()
fig.savefig(FIGS / 'zip_residual_map.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGS / 'zip_residual_map.pdf', bbox_inches='tight')
plt.show()